# CLIP evaluation — metrics for your report

## Before you run

1. **Checkpoints on this machine** (after HF download or copy):
   - `lightning_logs_clip/checkpoints/last.ckpt` (or your best `clip-epoch=...ckpt`)
2. **Test manifest** with labels: `data_raw/manifests/test_with_faces.csv`
3. **Images** must exist at paths in the CSV (use `--remap_from` / `--remap_to` if you moved from RunPod → Mac).

## mAP vs what we report

- **mAP (mean Average Precision)** is standard for **object detection** (COCO-style, multiple IoU thresholds).
- Your task is **binary image classification** (Real vs AI) with **RetinaFace** only for **cropping**, not for class mAP.
- We report **AP** = **AUC-PR** (`average_precision_score` / **AUC-PR**): ranking quality for the AI class.
- Plus **precision, recall, F1** at your chosen threshold **τ**, **AUC-ROC**, **accuracy**, **inference speed**.

## RetinaFace (face detector)

- To report **detection mAP**, you need a **benchmark with bounding-box ground truth** (not in this CSV).
- Optional report line: **% images with ≥1 face detected**, mean detection confidence — from a small script if needed.

---
Run the **next cell** after setting `PROJECT_ROOT` and checkpoint path.

In [ ]:
import os, sys, json
from pathlib import Path

_here = Path.cwd().resolve()
REPO = _here.parent if _here.name == "notebooks" else _here
os.environ["PROJECT_ROOT"] = os.environ.get("PROJECT_ROOT", str(REPO))
sys.path.insert(0, str(REPO))

# --- edit if needed ---
CKPT = REPO / "lightning_logs_clip/checkpoints/last.ckpt"
MAX_SAMPLES = 500   # None = full test set; use a number for a quick test
BATCH = 32
THRESHOLD = 0.5
# Mac vs RunPod paths in CSV:
REMAP_FROM = None  # e.g. "/workspace/AI_COMPUTER_VISION"
REMAP_TO = None    # e.g. str(REPO)

assert CKPT.is_file(), f"Missing checkpoint: {CKPT}"

from evaluation.evaluate_clip import main as eval_main

argv = [
    "evaluate_clip.py",
    "--checkpoint", str(CKPT),
    "--batch_size", str(BATCH),
    "--threshold", str(THRESHOLD),
    "--num_workers", "4",
    "--time_trial", "20",
]
if MAX_SAMPLES:
    argv += ["--max_samples", str(MAX_SAMPLES)]
if REMAP_FROM and REMAP_TO:
    argv += ["--remap_from", REMAP_FROM, "--remap_to", REMAP_TO]

sys.argv = argv
eval_main()

out_json = REPO / "results" / "evaluation_clip" / "clip_eval_results.json"
if out_json.is_file():
    print("\n--- Saved metrics ---")
    print(json.dumps(json.loads(out_json.read_text()), indent=2))

## CLI (same thing from a terminal)

```bash
cd /path/to/AI_COMPUTER_VISION
export PROJECT_ROOT="$PWD"
python3 evaluation/evaluate_clip.py \
  --checkpoint lightning_logs_clip/checkpoints/last.ckpt \
  --max_samples 2000 \
  --batch_size 32 \
  --threshold 0.5 \
  --time_trial 20
```

Results: `results/evaluation_clip/clip_eval_results.json`